# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [2]:
# Load raw data (file uses latin-1 encoding)
df = pd.read_csv('AviationData.csv', encoding='latin-1', low_memory=False)
print(df.shape)
display(df.head(3))
df.info()
print("\nNaN counts:")
print(df.isna().sum().sort_values(ascending=False).head(15))
display(df.describe())

(88889, 31)


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007


<class 'pandas.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  str    
 1   Investigation.Type      88889 non-null  str    
 2   Accident.Number         88889 non-null  str    
 3   Event.Date              88889 non-null  str    
 4   Location                88837 non-null  str    
 5   Country                 88663 non-null  str    
 6   Latitude                34382 non-null  str    
 7   Longitude               34373 non-null  str    
 8   Airport.Code            50132 non-null  str    
 9   Airport.Name            52704 non-null  str    
 10  Injury.Severity         87889 non-null  str    
 11  Aircraft.damage         85695 non-null  str    
 12  Aircraft.Category       32287 non-null  str    
 13  Registration.Number     87507 non-null  str    
 14  Make                    88826 non-null  str    
 

Schedule                  76307
Air.carrier               72241
FAR.Description           56866
Aircraft.Category         56602
Longitude                 54516
Latitude                  54507
Airport.Code              38757
Airport.Name              36185
Broad.phase.of.flight     27165
Publication.Date          13771
Total.Serious.Injuries    12510
Total.Minor.Injuries      11933
Total.Fatal.Injuries      11401
Engine.Type                7096
Report.Status              6384
dtype: int64


,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


**Observations:** 88,889 rows x 31 columns. `Aircraft.Category`, `Schedule`, `Air.carrier`, `FAR.Description`, lat/long and airport fields are mostly empty. Injury columns have ~12k NaNs each. Text columns have inconsistent casing and spelling variants (e.g. `UNK`/`Unk`/`Unknown`).

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [3]:
# Convert date and keep 1983 onwards (40-year max lifetime assumption)
df['Event.Date'] = pd.to_datetime(df['Event.Date'])
df = df[df['Event.Date'].dt.year >= 1983].copy()
print("1983+:", len(df))

# Professional builds only (drop amateur-built; drop the few NaNs since build status is unknown)
print(df['Amateur.Built'].value_counts(dropna=False))
df = df[df['Amateur.Built'] == 'No'].copy()
print("professional builds:", len(df))

# Keep full accident investigations only. 'Incident' records are minor events
# (no/low damage) and would bias injury/destruction rates downward.
df = df[df['Investigation.Type'] == 'Accident'].copy()
print("accidents only:", len(df))

# Make/Model are needed to identify plane types -> drop the few NaNs, standardise text
df = df.dropna(subset=['Make', 'Model']).copy()
df['Make'] = df['Make'].str.upper().str.strip()
df['Model'] = df['Model'].str.upper().str.strip()

# Aircraft.Category is ~66% NaN. Imputation: for a missing category, use the most
# common category recorded for the same Make + Model elsewhere in the data.
# We need a normalised Make/Model first, so the imputation happens in the Make/Model sections below.
print(df['Aircraft.Category'].value_counts(dropna=False).head(6))

1983+: 85289
Amateur.Built
No     76960
Yes     8228
NaN      101
Name: count, dtype: int64


professional builds: 76960
accidents only: 73286


Aircraft.Category
NaN             49461
Airplane        19923
Helicopter       2933
Glider            434
Balloon           197
Weight-Shift      139
Name: count, dtype: int64


**Filtering assumptions**
- Date >= 1983 (max 40-year plane lifetime, per the brief).
- `Amateur.Built == 'No'` (professional builds only).
- `Investigation.Type == 'Accident'` only: incidents are minor events and would understate risk.
- Aircraft category is imputed from Make+Model after those columns are cleaned (see below), then only `Airplane` is kept (helicopters, gliders, balloons etc. are outside the client's scope).

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [4]:
inj = ['Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured']

# Rows with no injury information at all cannot be used -> drop.
df = df.dropna(subset=inj, how='all').copy()
# Remaining NaNs mean "none recorded" for that severity -> 0
df[inj] = df[inj].fillna(0)

# Estimated number of people on board = sum of all injury categories
df['Total_Onboard'] = df[inj].sum(axis=1)
df = df[df['Total_Onboard'] > 0].copy()          # need a non-zero denominator

# Fraction of people on board that were fatally or seriously injured
df['FatalSerious_Frac'] = (df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']) / df['Total_Onboard']

print(len(df))
display(df[['Total_Onboard', 'FatalSerious_Frac']].describe())

72819


,Total_Onboard,FatalSerious_Frac
count,72819.000000,72819.000000
mean,4.530191,0.286003
std,21.699603,0.434093
min,1.000000,0.000000
25%,1.000000,0.000000
50%,2.000000,0.000000
75%,2.000000,1.000000
max,699.000000,1.000000


**Derived columns**
- `Total_Onboard` = fatal + serious + minor + uninjured (includes crew; the data has no separate passenger count).
- `FatalSerious_Frac` = (fatal + serious) / `Total_Onboard`, in [0, 1].
- NaN in an individual injury column is treated as 0 only when at least one other injury column is populated.

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [5]:
print(df['Aircraft.damage'].value_counts(dropna=False))

# Standardise: NaN / 'Unknown' -> NaN (kept in the data, excluded from destruction-rate calcs)
df['Aircraft.damage'] = df['Aircraft.damage'].replace('Unknown', np.nan)

# Derived: 1 = destroyed, 0 = substantial/minor damage, NaN = unknown
df['Destroyed'] = np.where(df['Aircraft.damage'].isna(), np.nan,
                           (df['Aircraft.damage'] == 'Destroyed').astype(float))
print(df['Destroyed'].value_counts(dropna=False))

Aircraft.damage
Substantial    55381
Destroyed      15362
NaN             1350
Minor            647
Unknown           79
Name: count, dtype: int64


Destroyed
0.0    56028
1.0    15362
NaN     1429
Name: count, dtype: int64


**Assumption:** `Destroyed` is 1 only for `Aircraft.damage == 'Destroyed'`. Unknown/missing damage stays NaN rather than being guessed, so those rows are excluded from destruction-rate calculations but still used for injury analysis.

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [6]:
import re

# Cleaning tasks (see markdown below):
def norm_make(m):
    m = re.sub(r'[.,]', '', m)
    m = re.sub(r'\b(INC|CORP|CORPORATION|CO|COMPANY|LTD|LLC|AIRCRAFT)\b', '', m)
    return re.sub(r'\s+', ' ', m).strip()

df['Make'] = df['Make'].map(norm_make)

merge = {'ROBINSON HELICOPTER': 'ROBINSON', 'GRUMMAN AMERICAN': 'GRUMMAN', 'GRUMMAN-SCHWEIZER': 'GRUMMAN',
         'CIRRUS DESIGN': 'CIRRUS', 'DEHAVILLAND': 'DE HAVILLAND', 'AIRBUS INDUSTRIE': 'AIRBUS',
         'DOUGLAS': 'MCDONNELL DOUGLAS', 'ERCOUPE (ENG & RESEARCH )': 'ERCOUPE',
         'BEECHCRAFT': 'BEECH', 'HUGHES HELICOPTERS': 'HUGHES',
         'ROCKWELL INTERNATIONAL': 'ROCKWELL', 'GRUMMAN ACFT ENG COR-SCHWEIZER': 'GRUMMAN',
         'GRUMMAN AMERICAN AVN': 'GRUMMAN', 'DIAMOND IND': 'DIAMOND', 'AMERICAN CHAMPION': 'CHAMPION'}
df['Make'] = df['Make'].replace(merge)
df.loc[df['Make'].str.startswith('EMBRAER'), 'Make'] = 'EMBRAER'
df.loc[df['Make'].str.startswith('BOMBARDIER'), 'Make'] = 'BOMBARDIER'
# 'Boeing' also appears as the maker of the Stearman Kaydet biplane (A75/B75/PT-17...): not a jet
stearman = df['Make'].str.startswith('BOEING') & df['Model'].str.match(r'^(A75|B75|D75|E75|PT|N2S|75[NL])')
df.loc[df['Make'].str.startswith('BOEING') & ~stearman, 'Make'] = 'BOEING'
df.loc[stearman, 'Make'] = 'BOEING STEARMAN'

print("unique makes after cleaning:", df['Make'].nunique())
print(df['Make'].value_counts().head(15))

unique makes after cleaning: 1522
Make
CESSNA         25369
PIPER          13945
BEECH           4934
BELL            2506
ROBINSON        1606
GRUMMAN         1499
MOONEY          1285
BELLANCA         973
BOEING           913
AIR TRACTOR      891
HUGHES           872
SCHWEIZER        769
AERONCA          606
CHAMPION         570
MAULE            570
Name: count, dtype: int64


**Make cleaning tasks**
1. Upper-case and strip whitespace.
2. Remove punctuation and corporate suffixes (INC, CORP, CO, LTD, AIRCRAFT...).
3. Separate Boeing-Stearman biplanes (models A75/B75/PT-17...) from Boeing jets, and merge spelling/naming variants of the same manufacturer (ROBINSON HELICOPTER -> ROBINSON, DEHAVILLAND -> DE HAVILLAND, AIRBUS INDUSTRIE -> AIRBUS, McDonnell Douglas/Douglas, Embraer/Bombardier sub-names, etc.).
4. Keep makes with a reasonable number of records (threshold = 50) **after** the airplane filter (applied at the end of the Model section, since category imputation needs Model).

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [7]:
# Model: standardised above (upper/strip, NaNs dropped). Remove spaces/hyphens/punctuation so
# '737-200', '737 200' and '737200' are treated as the same model.
df['Model_clean'] = df['Model'].str.replace(r'[^A-Z0-9]', '', regex=True)
df = df[df['Model_clean'] != ''].copy()

# Model labels are NOT unique across makes (e.g. '172' / '100' exist for several makes),
# so build a unique plane-type identifier: Make + Model
df['Aircraft_Type'] = df['Make'] + ' ' + df['Model_clean']
print("models:", df['Model_clean'].nunique(), "| unique Make+Model types:", df['Aircraft_Type'].nunique())

# Impute missing Aircraft.Category using the most common category of the same Aircraft_Type
known = df.dropna(subset=['Aircraft.Category'])
type_cat = known.groupby('Aircraft_Type')['Aircraft.Category'].agg(lambda s: s.mode().iloc[0])
df['Aircraft.Category'] = df['Aircraft.Category'].fillna(df['Aircraft_Type'].map(type_cat))
print(df['Aircraft.Category'].value_counts(dropna=False).head(6))

# Keep airplanes only (unmatched/NaN category rows are dropped)
df = df[df['Aircraft.Category'] == 'Airplane'].copy()

# Keep makes with >= 50 records
make_counts = df['Make'].value_counts()
df = df[df['Make'].isin(make_counts[make_counts >= 50].index)].copy()
print("rows:", len(df), "| makes kept:", df['Make'].nunique())

models: 5594 | unique Make+Model types: 7119


Aircraft.Category
Airplane        59250
Helicopter       6836
NaN              5166
Glider            910
Balloon           350
Weight-Shift      139
Name: count, dtype: int64
rows: 56126 | makes kept: 45


**Model cleaning:** NaN models dropped; model strings normalised; because the same model label occurs under different makes, `Aircraft_Type = Make + Model` is used as the unique plane identifier. Missing `Aircraft.Category` is imputed from other records of the same `Aircraft_Type`; rows that cannot be matched are dropped.

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [8]:
# Engine.Type: unify unknown labels, merge geared turbofan with turbofan
df['Engine.Type'] = df['Engine.Type'].replace({'UNK': 'Unknown', 'LR': 'Unknown', 'NONE': 'Unknown',
                                               'Geared Turbofan': 'Turbo Fan'}).fillna('Unknown')
# Weather.Condition: UNK / Unk / NaN -> Unknown
df['Weather.Condition'] = df['Weather.Condition'].replace({'UNK': 'Unknown', 'Unk': 'Unknown'}).fillna('Unknown')
# Number.of.Engines: 0 engines is invalid for an airplane -> NaN
df['Number.of.Engines'] = df['Number.of.Engines'].replace(0, np.nan)
# Phase / purpose of flight: NaN -> Unknown
df['Broad.phase.of.flight'] = df['Broad.phase.of.flight'].fillna('Unknown')
df['Purpose.of.flight'] = df['Purpose.of.flight'].fillna('Unknown')

# Small-category groupings: very rare engine counts (>4) -> NaN (likely data entry errors)
df.loc[df['Number.of.Engines'] > 4, 'Number.of.Engines'] = np.nan

for c in ['Engine.Type', 'Weather.Condition', 'Number.of.Engines', 'Purpose.of.flight', 'Broad.phase.of.flight']:
    print(df[c].value_counts(dropna=False).head(10), '\n')

Engine.Type
Reciprocating    49527
Unknown           3341
Turbo Prop        2211
Turbo Fan          870
Turbo Jet          165
Turbo Shaft         12
Name: count, dtype: int64 

Weather.Condition
VMC        49355
IMC         4416
Unknown     2355
Name: count, dtype: int64 

Number.of.Engines
1.0    46903
2.0     6814
NaN     2247
4.0       82
3.0       80
Name: count, dtype: int64 

Purpose.of.flight
Personal               33525
Instructional           7802
Unknown                 5517
Aerial Application      3280
Business                2691
Positioning              891
Other Work Use           496
Ferry                    479
Aerial Observation       372
Executive/corporate      279
Name: count, dtype: int64 

Broad.phase.of.flight
Unknown        16100
Landing        11084
Takeoff         8228
Cruise          6688
Maneuvering     4346
Approach        4203
Taxi            1307
Climb           1275
Descent         1201
Go-around       1112
Name: count, dtype: int64 



**Cleaning tasks:** unified "unknown" labels (UNK/Unk/NaN), merged Geared Turbofan into Turbo Fan, set impossible engine counts (0 or >4 on an airplane) to NaN, filled categorical NaNs with `Unknown` so those rows are retained but can be excluded in analysis.

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [9]:
# Drop columns with > 50% NaN
nan_frac = df.isna().mean().sort_values(ascending=False)
print(nan_frac.head(12))
to_drop = nan_frac[nan_frac > 0.5].index.tolist()
print("dropping:", to_drop)
df = df.drop(columns=to_drop)
print(df.shape)

Schedule             0.891334
Air.carrier          0.832555
FAR.Description      0.692513
Longitude            0.614279
Latitude             0.614190
Airport.Code         0.393632
Airport.Name         0.365517
Publication.Date     0.167587
Report.Status        0.055304
Number.of.Engines    0.040035
Aircraft.damage      0.014271
Destroyed            0.014271
dtype: float64
dropping: ['Schedule', 'Air.carrier', 'FAR.Description', 'Longitude', 'Latitude']
(56126, 31)


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [10]:
df.to_csv('Aviation_Cleaned.csv', index=False)
print("saved", df.shape)

saved (56126, 31)
